<a href="https://colab.research.google.com/github/vggd18/tlc-trip-spark-aws/blob/main/notebook/PySpark_AWS.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🏗️ Building a Data Lakehouse: Do Lixo ao Luxo com PySpark & AWS

Bem-vindo a este laboratório prático! Aqui, vamos construir um pipeline de Engenharia de Dados completo, transformando dados brutos de Táxis de Nova York em insights de negócio valiosos.

**Nossa Jornada (Arquitetura Medallion):**
1.  **Landing Zone:** Download de dados brutos (Parquet).
2.  **🥉 Bronze Layer:** Ingestão no formato Delta Lake (Histórico Fiel).
3.  **🥈 Silver Layer:** Limpeza, tipagem e enriquecimento (Fonte da Verdade).
4.  **🥇 Gold Layer:** Agregações e métricas de negócio (Pronto para BI).

---
**Stack Tecnológica:** Python, PySpark, Delta Lake, AWS S3 (Simulado).

## 🛠️ 1. Configuração do Ambiente e Landing Zone

Nesta etapa, preparamos o terreno. Vamos instalar as dependências do **Delta Lake** (necessário para as transações ACID) e simular nossa "Landing Zone" baixando os dados brutos de viagens de táxi de 2019.

> **Obs:** Estamos baixando 12 meses de dados reais para garantir volume e complexidade para o Spark processar.

In [1]:
!pip install pyspark==3.5.1 delta-spark==3.2.0 -q

In [2]:
import os

output_dir = "./nyc_taxi_2019"
os.makedirs(output_dir, exist_ok=True)

base_url = "https://d37ci6vzurychx.cloudfront.net/trip-data"

print(f"Iniciando download dos 12 meses de 2019 para: {output_dir}")

for month in range(1, 13):
  month_str = f"{month:02d}"
  file_name = f"yellow_tripdata_2019-{month_str}.parquet"
  url = f"{base_url}/{file_name}"
  output_path = f"{output_dir}/{file_name}"

  print(f"[{month}/12] Baixando: {file_name}...")

  exit_status = os.system(f'wget -q --show-progress "{url}" -O "{output_path}"')

  if exit_status != 0:
    print(f"Erro ao baixar o mês {month_str}")

print("\n Download completo de 2019!")

Iniciando download dos 12 meses de 2019 para: ./nyc_taxi_2019
[1/12] Baixando: yellow_tripdata_2019-01.parquet...
[2/12] Baixando: yellow_tripdata_2019-02.parquet...
[3/12] Baixando: yellow_tripdata_2019-03.parquet...
[4/12] Baixando: yellow_tripdata_2019-04.parquet...
[5/12] Baixando: yellow_tripdata_2019-05.parquet...
[6/12] Baixando: yellow_tripdata_2019-06.parquet...
[7/12] Baixando: yellow_tripdata_2019-07.parquet...
[8/12] Baixando: yellow_tripdata_2019-08.parquet...
[9/12] Baixando: yellow_tripdata_2019-09.parquet...
[10/12] Baixando: yellow_tripdata_2019-10.parquet...
[11/12] Baixando: yellow_tripdata_2019-11.parquet...
[12/12] Baixando: yellow_tripdata_2019-12.parquet...

 Download completo de 2019!


## ⚡ 2. Inicializando o Cluster Spark

Aqui configuramos a `SparkSession`, o ponto de entrada da nossa aplicação. Note que estamos injetando as configurações do **Delta Lake** (`io.delta.sql.DeltaSparkSessionExtension`) para habilitar funcionalidades como *Schema Enforcement* e *Time Travel*.

In [3]:
import pyspark
from pyspark.sql import SparkSession
from pyspark.sql.functions import current_timestamp, input_file_name
from delta import *

builder = SparkSession.builder \
  .appName("DeltaBronzeLayer") \
  .master("local[*]") \
  .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension") \
  .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog") \
  .config("spark.jars.packages", "io.delta:delta-spark_2.12:3.0.0")

spark = configure_spark_with_delta_pip(builder).getOrCreate()

print("Spark com Delta Lake iniciado com sucesso!")

Spark com Delta Lake iniciado com sucesso!


## 🥉 3. Camada Bronze (Raw Ingestion)

A Camada Bronze tem um objetivo claro: **Ingestão com alta fidelidade**.

Trazemos os dados da Landing Zone para dentro do Lakehouse convertendo-os para o formato **Delta**.
* **Schema:** Definimos um esquema explícito para evitar erros de leitura.
* **Metadados:** Adicionamos colunas de controle (`ingestion_date`, `source_file`).
* **Particionamento:** Organizamos os dados fisicamente por `Ano` e `Mês` para otimizar leituras futuras.

In [4]:
df_2019 = spark.read.parquet("./nyc_taxi_2019/yellow_tripdata_2019-01.parquet")

print(f"Total de registros em 2019: {df_2019.count():,}")
df_2019.printSchema()
df_2019.show(5)

df_2019.describe().show()

Total de registros em 2019: 7,696,617
root
 |-- VendorID: long (nullable = true)
 |-- tpep_pickup_datetime: timestamp_ntz (nullable = true)
 |-- tpep_dropoff_datetime: timestamp_ntz (nullable = true)
 |-- passenger_count: double (nullable = true)
 |-- trip_distance: double (nullable = true)
 |-- RatecodeID: double (nullable = true)
 |-- store_and_fwd_flag: string (nullable = true)
 |-- PULocationID: long (nullable = true)
 |-- DOLocationID: long (nullable = true)
 |-- payment_type: long (nullable = true)
 |-- fare_amount: double (nullable = true)
 |-- extra: double (nullable = true)
 |-- mta_tax: double (nullable = true)
 |-- tip_amount: double (nullable = true)
 |-- tolls_amount: double (nullable = true)
 |-- improvement_surcharge: double (nullable = true)
 |-- total_amount: double (nullable = true)
 |-- congestion_surcharge: double (nullable = true)
 |-- airport_fee: integer (nullable = true)

+--------+--------------------+---------------------+---------------+-------------+--------

In [5]:
from pyspark.sql.types import *

schema = StructType([
  StructField("VendorID", LongType(), True),
  StructField("tpep_pickup_datetime", TimestampType(), True),
  StructField("tpep_dropoff_datetime", TimestampType(), True),
  StructField("passenger_count", DoubleType(), True),
  StructField("trip_distance", DoubleType(), True),
  StructField("RatecodeID", DoubleType(), True),
  StructField("store_and_fwd_flag", StringType(), True),
  StructField("PULocationID", LongType(), True),
  StructField("DOLocationID", LongType(), True),
  StructField("payment_type", LongType(), True),
  StructField("fare_amount", DoubleType(), True),
  StructField("extra", DoubleType(), True),
  StructField("mta_tax", DoubleType(), True),
  StructField("tip_amount", DoubleType(), True),
  StructField("tolls_amount", DoubleType(), True),
  StructField("improvement_surcharge", DoubleType(), True),
  StructField("total_amount", DoubleType(), True),
  StructField("congestion_surcharge", DoubleType(), True),
  StructField("airport_fee", IntegerType(), True)
])

raw_path = "/content/nyc_taxi_2019/*.parquet"
bronze_path = "/content/lakehouse/bronze/taxi_trips"

print("1. Lendo dados Raw...")
df_raw = spark.read \
  .schema(schema) \
  .option("header", True) \
  .option("mergeSchema", True) \
  .parquet(raw_path)

1. Lendo dados Raw...


In [6]:
print("2. Adicionando metadados de ingestão...")
df_bronze = df_raw \
  .withColumn("ingestion_date", current_timestamp()) \
  .withColumn("source_file", input_file_name())

2. Adicionando metadados de ingestão...


In [7]:
from pyspark.sql.functions import year, month
print("3. Partcionando camada Bronze...")

df_bronze_partitioned = df_bronze \
  .withColumn("year", year("tpep_pickup_datetime")) \
  .withColumn("month", month("tpep_pickup_datetime"))

3. Partcionando camada Bronze...


In [8]:
print(f"4. Escrevendo Camada Bronze em Delta: {bronze_path}")
df_bronze_partitioned.write \
  .format("delta") \
  .mode("overwrite") \
  .option("partitionOverwriteMode", "dynamic") \
  .partitionBy("year", "month") \
  .save(bronze_path)

print("✅ Carga Bronze com Particionamento Dinâmico Finalizada!")

4. Escrevendo Camada Bronze em Delta: /content/lakehouse/bronze/taxi_trips
✅ Carga Bronze com Particionamento Dinâmico Finalizada!


In [9]:
from pyspark.sql.functions import col

df_bronze = spark.read.format("delta").load(bronze_path)

print("--- Viajantes do Tempo na Bronze ---")
df_bronze.groupBy("year").count().orderBy("year").show()

--- Viajantes do Tempo na Bronze ---
+----+--------+
|year|   count|
+----+--------+
|2001|       3|
|2002|      13|
|2003|       6|
|2008|     202|
|2009|     402|
|2010|       1|
|2015|       1|
|2018|     366|
|2019|84597002|
|2020|     428|
|2026|       1|
|2029|       4|
|2033|       3|
|2038|       4|
|2041|       1|
|2058|       3|
|2066|       1|
|2088|       2|
|2090|       1|
+----+--------+



### 📊 Data Profiling (Raio-X dos Dados)

Antes de transformar, precisamos conhecer o tamanho do nosso desafio. Vamos calcular o volume físico em disco e estimar o consumo em memória RAM para processamento.

In [10]:
import os

num_cols = len(df_2019.columns)
num_rows = df_2019.count()

def get_size(start_path):
  total_size = 0
  for dirpath, dirnames, filenames in os.walk(start_path):
    for f in filenames:
      fp = os.path.join(dirpath, f)
      if not os.path.islink(fp):
        total_size += os.path.getsize(fp)
  return total_size

bytes_disk = get_size("/content/nyc_taxi_2019")
gb_disk = bytes_disk / (1024**3)
gb_memory_estimate = gb_disk * 5

print(f"--- Estatísticas do Dataset (Ano 2019) ---")
print(f"📋 Colunas: {num_cols}")
print(f"📊 Linhas:  {num_rows:,}")
print(f"💾 Tamanho em Disco (Parquet Comprimido): {gb_disk:.2f} GB")
print(f"🧠 Estimativa em Memória (Se fosse CSV/Pandas): ~{gb_memory_estimate:.2f} GB")

--- Estatísticas do Dataset (Ano 2019) ---
📋 Colunas: 19
📊 Linhas:  7,696,617
💾 Tamanho em Disco (Parquet Comprimido): 1.16 GB
🧠 Estimativa em Memória (Se fosse CSV/Pandas): ~5.79 GB


## 🥈 4. Camada Silver (Cleansed & Enriched)

Aqui é onde a mágica da Engenharia de Dados acontece. Transformaremos dados "sujos" em informações confiáveis.

**Nossas tarefas de refinamento:**
1.  **Rename:** Padronização de colunas para `snake_case`.
2.  **Data Quality:** Análise e remoção de colunas com excesso de nulos.
3.  **Type Casting:** Garantia de tipagem forte (ex: IDs como Inteiros).
4.  **Enrichment:** Mapeamento de códigos (ex: `1` -> `Credit Card`) e criação de novas features (duração da viagem).
5.  **Business Rules:** Filtragem de dados inválidos (viagens negativas, passageiros zerados).

In [11]:
from pyspark.sql.functions import col, when, round, unix_timestamp

bronze_path = "./lakehouse/bronze/taxi_trips"
silver_path = "./lakehouse/silver/taxi_trips"

print("🚀 Iniciando Pipeline Silver (Curated)...")

df = spark.read.format("delta").load(bronze_path)

🚀 Iniciando Pipeline Silver (Curated)...


### 1. RENAME (Padronização para snake_case)

In [12]:
df_step1 = df \
  .withColumnRenamed("VendorID", "vendor_id") \
  .withColumnRenamed("tpep_pickup_datetime", "pickup_datetime") \
  .withColumnRenamed("tpep_dropoff_datetime", "dropoff_datetime") \
  .withColumnRenamed("PULocationID", "pickup_location_id") \
  .withColumnRenamed("DOLocationID", "dropoff_location_id") \
  .withColumnRenamed("RatecodeID", "rate_code_id")

### 2. DROP (Null Count / Limpeza Técnica)

In [13]:
from pyspark.sql.functions import col, count, when

df_target = df_step1
total_rows = df_target.count()

print(f"📊 Total de Linhas: {total_rows:,}")

null_counts = df_target.select([
  count(when(col(c).isNull(), c)).alias(c)
  for c in df_target.columns
]).collect()[0].asDict()

threshold = 0.70
cols_to_drop = []

print("\n--- Análise de Nulos ---")
for col_name, null_count in null_counts.items():
  null_pct = null_count / total_rows

  if null_pct > threshold:
    print(f"❌ {col_name}: {null_pct:.1%} de nulos (SERÁ REMOVIDA)")
    cols_to_drop.append(col_name)
  elif null_pct > 0:
    print(f"⚠️ {col_name}: {null_pct:.1%} de nulos (Mantida)")

if cols_to_drop:
  df_clean_nulls = df_target.drop(*cols_to_drop)
  print(f"\n🧹 Colunas Removidas: {cols_to_drop}")
else:
  df_clean_nulls = df_target
  print("\n✅ Nenhuma coluna excedeu o limite de nulos.")

df_clean_nulls.printSchema()

📊 Total de Linhas: 84,598,444

--- Análise de Nulos ---
⚠️ passenger_count: 0.5% de nulos (Mantida)
⚠️ rate_code_id: 0.5% de nulos (Mantida)
⚠️ store_and_fwd_flag: 0.5% de nulos (Mantida)
⚠️ congestion_surcharge: 6.3% de nulos (Mantida)
❌ airport_fee: 100.0% de nulos (SERÁ REMOVIDA)

🧹 Colunas Removidas: ['airport_fee']
root
 |-- vendor_id: long (nullable = true)
 |-- pickup_datetime: timestamp (nullable = true)
 |-- dropoff_datetime: timestamp (nullable = true)
 |-- passenger_count: double (nullable = true)
 |-- trip_distance: double (nullable = true)
 |-- rate_code_id: double (nullable = true)
 |-- store_and_fwd_flag: string (nullable = true)
 |-- pickup_location_id: long (nullable = true)
 |-- dropoff_location_id: long (nullable = true)
 |-- payment_type: long (nullable = true)
 |-- fare_amount: double (nullable = true)
 |-- extra: double (nullable = true)
 |-- mta_tax: double (nullable = true)
 |-- tip_amount: double (nullable = true)
 |-- tolls_amount: double (nullable = true)
 |-

In [14]:
df_step2 = df_clean_nulls.dropna(subset=[
  "pickup_datetime",
  "dropoff_datetime",
  "total_amount"
])

### 3. TYPE CONVERT (Tipagem Forte)

In [15]:
df_step3 = df_step2 \
  .withColumn("passenger_count", col("passenger_count").cast("int")) \
  .withColumn("payment_type", col("payment_type").cast("int")) \
  .withColumn("rate_code_id", col("rate_code_id").cast("int")) \
  .withColumn("pickup_location_id", col("pickup_location_id").cast("int")) \
  .withColumn("dropoff_location_id", col("dropoff_location_id").cast("int"))

### 4. CATEGORICAL VALUES (Padronização de Domínio)

In [16]:
from pyspark.sql.functions import udf, col
from pyspark.sql.types import StringType

payment_dict = {
  1: "Credit Card", 2: "Cash", 3: "No Charge",
  4: "Dispute", 5: "Unknown", 6: "Voided Trip"
}

def payment_map(code):
  return payment_dict.get(code, "Unknown")

payment_map_udf = udf(lambda z: payment_map(z), StringType())

df_step4_1 = df_step3.withColumn(
  "payment_type_desc",
  payment_map_udf(col("payment_type"))
)

In [17]:
rate_code_map = {
  1: "Standard", 2: "JFK", 3: "Newark", 4: "Nassau/Westchester",
  5: "Negotiated", 6: "Group Ride", 99: "Unknown"
}

def rate_code_map(code):
  return payment_dict.get(code, "Unknown")

rate_code_map_udf = udf(lambda z: rate_code_map(z), StringType())

df_step4_2 = df_step4_1.withColumn(
  "payment_type_desc",
  rate_code_map_udf(col("payment_type"))
)


In [18]:
vendor_map = {
  1: "Creative Mobile Tech", 2: "Curb Mobility",
  6: "Myle Tech", 7: "Helix"
}

def vendor_map(code):
  return payment_dict.get(code, "Unknown")

vendor_map_udf = udf(lambda z: vendor_map(z), StringType())

df_step4_3 = df_step4_2.withColumn(
  "payment_type_desc",
  vendor_map_udf(col("payment_type"))
)

In [19]:
df_step4 = df_step4_3.withColumn(
  "store_and_fwd_flag",
  when(col("store_and_fwd_flag") == "Y", "Y").otherwise("N")
)

### 5. COLUNAS A PARTIR DE OUTRAS (Feature Engineering)

In [20]:
from pyspark.sql.functions import col, when, round, unix_timestamp, date_format, hour

df_step5 = df_step4 \
  .withColumn(
    "trip_duration_minutes",
    round((unix_timestamp("dropoff_datetime") - unix_timestamp("pickup_datetime")) / 60, 2)
  ) \
  .withColumn("day_of_week", date_format("pickup_datetime", "EEEE")) \
  .withColumn("hour", hour("pickup_datetime"))

### 6. AJUSTE NUMÉRICO (Regras de Negócio / Data Quality)

In [21]:
df_silver = df_step5 \
  .filter(col("year") == 2019) \
  .filter(col("total_amount") > 0) \
  .filter(col("trip_duration_minutes") > 0) \
  .filter(col("passenger_count") > 0) \
  .withColumn("total_amount", round(col("total_amount"), 2)) \
  .withColumn("fare_amount", round(col("fare_amount"), 2))

### 💾 Persistência da Silver

Salvamos o resultado limpo em formato Delta. Note a opção `overwriteSchema`: como alteramos a estrutura dos dados (removemos colunas e mudamos tipos), precisamos autorizar o Delta Lake a atualizar o esquema da tabela.

In [22]:
print("💾 Gravando Silver particionada...")

df_silver.write \
  .format("delta") \
  .mode("overwrite") \
  .partitionBy("year", "month") \
  .option("overwriteSchema", "true") \
  .save(silver_path)

print("✅ Pipeline Silver Concluído!")

print("\n--- Amostra Silver (Schema Limpo) ---")
spark.read.format("delta").load(silver_path).select(
  "pickup_datetime", "passenger_count", "total_amount", "trip_duration_minutes"
).show(5)

💾 Gravando Silver particionada...
✅ Pipeline Silver Concluído!

--- Amostra Silver (Schema Limpo) ---
+-------------------+---------------+------------+---------------------+
|    pickup_datetime|passenger_count|total_amount|trip_duration_minutes|
+-------------------+---------------+------------+---------------------+
|2019-02-21 11:22:37|              1|        10.3|                 8.92|
|2019-02-28 19:52:45|              1|        18.3|                 9.15|
|2019-02-21 11:39:35|              1|        55.3|                 5.47|
|2019-02-28 23:59:46|              1|        10.8|                 4.18|
|2019-02-21 11:48:40|              1|       28.56|                 6.73|
+-------------------+---------------+------------+---------------------+
only showing top 5 rows



In [ ]:
df_silver.describe().show()

## 🥇 5. Camada Gold (Business Aggregation)

A Camada Gold é focada no consumidor final (Analistas e Dashboards). Os dados aqui são altamente agregados e organizados para responder perguntas de negócio.

**Foco: Performance Financeira** 💰
Vamos calcular KPIs como:
* Receita Total e Ticket Médio.
* Comportamento de Gorjetas (% médio).
* Share de Receita por dia da semana (usando *Window Functions*).

In [ ]:
from pyspark.sql.window import Window
from pyspark.sql.functions import sum, avg, count, round, col, when, desc

gold_fin_path = "/content/lakehouse/gold/financial_performance"

print("🚀 Construindo Gold Financeira (Revenue & Tipping)...")

🚀 Construindo Gold Financeira (Revenue & Tipping)...


In [ ]:
df_gold_input = df_silver.withColumn(
  "tip_percentage",
  when(col("fare_amount") > 0,
  col("tip_amount") / col("fare_amount")
  ).otherwise(0.0)
).withColumn(
  "is_credit_card",
  when(col("payment_type") == 1, "Credit Card")
  .when(col("payment_type") == 2, "Cash")
  .otherwise("Other")
)

In [ ]:
df_gold_agg = df_gold_input.groupBy("day_of_week", "is_credit_card") \
  .agg(
    count("*").alias("total_rides"),
    round(sum("total_amount"), 2).alias("total_revenue"),
    round(avg("total_amount"), 2).alias("avg_ticket"),
    round(avg("tip_percentage") * 100, 2).alias("avg_tip_pct"),
    round(sum("tolls_amount"), 2).alias("total_tolls"),
    round(sum("mta_tax") + sum("improvement_surcharge"), 2).alias("total_taxes")
  )

In [ ]:
window_daily = Window.partitionBy("day_of_week")

df_gold_refined = df_gold_agg.withColumn(
  "revenue_share_daily",
  round(col("total_revenue") / sum("total_revenue").over(window_daily) * 100, 2)
)

In [ ]:
df_gold_refined \
  .orderBy(desc("total_revenue")) \
  .write \
  .format("delta") \
  .mode("overwrite") \
  .save(gold_fin_path)

print("✅ Gold Financeira gerada!")
df_gold_refined.show(10)

✅ Gold Financeira gerada!
+-----------+--------------+-----------+--------------+----------+-----------+-----------+-----------+-------------------+
|day_of_week|is_credit_card|total_rides| total_revenue|avg_ticket|avg_tip_pct|total_tolls|total_taxes|revenue_share_daily|
+-----------+--------------+-----------+--------------+----------+-----------+-----------+-----------+-------------------+
|     Friday|         Other|      70701|    3063121.91|     43.33|       0.02|   30238.32|   54938.15|               1.25|
|     Friday|   Credit Card|    9131801|1.8669951554E8|     20.44|      26.83| 3820221.15| 7277153.32|              76.17|
|     Friday|          Cash|    3413185|  5.53471257E7|     16.22|        0.0|  994578.14| 2724548.85|              22.58|
|     Monday|         Other|      58365|    2195674.56|     37.62|       0.02|   27006.65|   45182.29|               1.05|
|     Monday|   Credit Card|    7691585|1.5895083308E8|     20.67|       28.1| 3741433.64| 6130699.67|           

### 📈 Analytics & Reporting

Com a tabela Gold pronta e registrada no catálogo, podemos atuar como Analistas de Dados e executar consultas SQL ad-hoc para extrair insights imediatos sobre a lucratividade da frota.

In [ ]:
spark.read.format("delta").load(gold_fin_path).createOrReplaceTempView("gold_financial")

print("💰 Relatório de Lucratividade (SQL):")

spark.sql("""
    SELECT
        day_of_week,
        is_credit_card,
        total_rides,
        avg_ticket,
        avg_tip_pct as generosity_index,
        revenue_share_daily as market_share_pct
    FROM gold_financial
    WHERE day_of_week IN ('Friday', 'Saturday') -- Fim de semana
    ORDER BY total_revenue DESC
""").show()

💰 Relatório de Lucratividade (SQL):
+-----------+--------------+-----------+----------+----------------+----------------+
|day_of_week|is_credit_card|total_rides|avg_ticket|generosity_index|market_share_pct|
+-----------+--------------+-----------+----------+----------------+----------------+
|     Friday|   Credit Card|    9131801|     20.44|           26.83|           76.17|
|   Saturday|   Credit Card|    8161506|     19.13|           27.62|           73.47|
|     Friday|          Cash|    3413185|     16.22|             0.0|           22.58|
|   Saturday|          Cash|    3554558|     15.55|             0.0|            26.0|
|     Friday|         Other|      70701|     43.33|            0.02|            1.25|
|   Saturday|         Other|      69102|     16.42|            0.01|            0.53|
+-----------+--------------+-----------+----------+----------------+----------------+

